In [1]:
import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import Normalizer
from sklearn.compose import ColumnTransformer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import InputLayer
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

I0000 00:00:1784844888.321997   12951 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1784844888.361571   12951 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1784844889.699366   12951 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
#Steps 1-10 
dataset = pd.read_csv("Life_expectancy.csv")
print(dataset.head())
print(dataset.describe())
dataset = dataset.drop(["Country"], axis = 1)
labels = dataset.iloc[:, -1]
features = dataset.iloc[:, 0:-1]
features = pd.get_dummies(features)
features_train, features_test, labels_train, labels_test = train_test_split(features, labels, test_size = 0.20, random_state = 23)
numerical_features = features.select_dtypes(include=['float64', 'int64'])
numerical_columns = numerical_features.columns
ct = ColumnTransformer([("only numeric", StandardScaler(), numerical_columns)], remainder='passthrough')
features_train_scaled = ct.fit_transform(features_train)
features_test = ct.transform(features_test)

       Country  Year      Status  Adult Mortality  infant deaths  Alcohol  \
0  Afghanistan  2015  Developing            263.0             62     0.01   
1  Afghanistan  2014  Developing            271.0             64     0.01   
2  Afghanistan  2013  Developing            268.0             66     0.01   
3  Afghanistan  2012  Developing            272.0             69     0.01   
4  Afghanistan  2011  Developing            275.0             71     0.01   

   percentage expenditure  Hepatitis B  Measles    BMI   ...  \
0               71.279624         65.0      1154   19.1  ...   
1               73.523582         62.0       492   18.6  ...   
2               73.219243         64.0       430   18.1  ...   
3               78.184215         67.0      2787   17.6  ...   
4                7.097109         68.0      3013   17.2  ...   

   Total expenditure  Diphtheria    HIV/AIDS         GDP  Population  \
0               8.16         65.0        0.1  584.259210  33736494.0   
1       

In [3]:
#Steps 11-16 
my_model = Sequential()
input = InputLayer(input_shape = (features.shape[1], ))
my_model.add(input)
my_model.add(Dense(64, activation = "relu"))
my_model.add(Dense(1))

/home/plewis/.local/lib/python3.10/site-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


In [4]:
#Steps 17-18 
opt = Adam(learning_rate = 0.01)
my_model.compile(loss = 'mse', metrics = ['mae'], optimizer = opt)
print(my_model.summary())

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         1,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,473 (5.75 KB)

 Trainable params: 1,473 (5.75 KB)

 Non-trainable params: 0 (0.00 B)

None


In [5]:
#Steps 19-21 
my_model.fit(features_train_scaled, labels_train, epochs = 40, batch_size = 1, verbose = 1)
res_mse, res_mae = my_model.evaluate(features_test, labels_test, verbose = 0)
print(res_mse, res_mae)

Epoch 1/40


   1/2350 ━━━━━━━━━━━━━━━━━━━━ 16:42 427ms/step - loss: 5728.6450 - mae: 75.6878

  26/2350 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 4701.3460 - mae: 68.1820     

  50/2350 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 4479.6806 - mae: 66.4663

  75/2350 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 4135.1242 - mae: 63.3402

...

2257/2350 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 8.1852 - mae: 2.0736

2286/2350 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 8.1844 - mae: 2.0737

2320/2350 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 8.1833 - mae: 2.0738

2350/2350 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 8.0440 - mae: 2.0761


6.978076934814453 1.863525152206421
